# Sensitivity Analysis of ODE Market-Making Model

Global sensitivity analysis (Sobol indices) and forward uncertainty quantification
for the 2-currency (USD, EUR) ODE approximation.

See `notes/sensitivity-analysis/setup.md` for methodology documentation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from tqdm import tqdm
import time

from src import build_paper_example_params, restrict_currencies, run_multicurrency_mm
from src.model import ModelParams, PairParams, TierParams, BP
from src.hamiltonian import logistic_f

## Parameter setup

8 parameters varied over uniform ranges around the paper's nominal values (Table 1, EUR/USD).

In [ ]:
PARAM_NAMES = [
    r"$\sigma_{EUR}$", r"$\gamma$", r"$\lambda_{scale}$",
    r"$\alpha_1$", r"$\alpha_2$",
    r"$\beta_1$", r"$\beta_2$", r"$\eta$"
]

PARAM_LABELS = [
    "sigma_EUR", "gamma", "lambda_scale",
    "alpha_1", "alpha_2", "beta_1", "beta_2", "eta"
]

# Nominal values in human-readable units
# sigma: bps, gamma: 1/M$, lambda_scale: multiplier,
# alpha: dimensionless, beta: 1/bps (paper units), eta: bps
NOMINAL = np.array([80.0, 20.0, 1.0, -1.9, -0.3, 11.0, 3.5, 1e-5])

RANGES = np.array([
    [56.0, 104.0],       # sigma_EUR (bps), +-30%
    [10.0, 40.0],        # gamma, +-50%
    [0.5, 1.5],          # lambda_scale, +-50%
    [-2.5, -1.3],        # alpha_1, +-30%
    [-0.9, 0.3],         # alpha_2, +-100%
    [7.7, 14.3],         # beta_1 (1/bps), +-30%
    [2.45, 4.55],        # beta_2 (1/bps), +-30%
    [0.5e-5, 2.0e-5],    # eta (bps), +-50%
])

# Set A: policy QoIs
# Set B: economic QoIs
QOI_NAMES = [
    # Set A
    r"$\delta^*_{T1}(0) - \delta^*_{T2}(0)$ [bps]",
    r"$\delta^*(y{=}10) - \delta^*(y{=}0)$ [bps]",
    r"$\xi^*(y{=}10)$ [M\$/day]",
    # Set B
    r"$R(y{=}0)$ [\$/day]",
    r"$L(\xi^*(y{=}10))$ [\$/day]",
    r"$R(y{=}10) - L(\xi^*)$ [\$/day]",
]

N_PARAMS = len(PARAM_NAMES)
N_QOIS = len(QOI_NAMES)
N_QOIS_A = 3  # first 3 are Set A, last 3 are Set B

## Model evaluation

Map the 8-dimensional parameter vector to 6 scalar quantities of interest
via the Riccati ODE solver.

**Set A — Policy QoIs:**
1. Tier spread differential $\delta^*_{T1}(y{=}0) - \delta^*_{T2}(y{=}0)$ at $z = 1$ M\$
2. Inventory skew $\delta^*(y{=}10) - \delta^*(y{=}0)$ (tier 1, $z = 1$ M\$)
3. Optimal hedge rate $\xi^*(y{=}10)$

**Set B — Economic QoIs:**
4. Expected revenue rate $R(y{=}0) = \sum_{\text{dir}} \sum_t \sum_j \lambda_j f(\delta^*_{t,j}) \delta^*_{t,j} z_j$
5. Hedging cost rate $L(\xi^*(y{=}10)) = \psi|\xi^*| + \eta (\xi^*)^2$
6. Net revenue rate $R(y{=}10) - L(\xi^*(y{=}10))$

In [ ]:
# Cache base parameters to avoid rebuilding every call
_base = restrict_currencies(build_paper_example_params(), ["USD", "EUR"])
_base_pp = _base.pairs[("EUR", "USD")]


def build_modified_params(params):
    """Build 2-currency (USD, EUR) ModelParams from parameter vector.

    params[0]: sigma_EUR in bps
    params[1]: gamma (risk aversion)
    params[2]: lambda_scale (multiplier on all arrival rates)
    params[3]: alpha_1 (logistic shift, aggressive tier)
    params[4]: alpha_2 (logistic shift, passive tier)
    params[5]: beta_1 (logistic slope, aggressive, in 1/bps paper units)
    params[6]: beta_2 (logistic slope, passive, in 1/bps paper units)
    params[7]: eta (execution cost quadratic, in bps)
    """
    sigma_eur, gamma, lam_scale, a1, a2, b1, b2, eta = params

    tiers = [
        TierParams(alpha=a1, beta=b1 * 1e4),
        TierParams(alpha=a2, beta=b2 * 1e4),
    ]

    pp = PairParams(
        pair=("EUR", "USD"),
        sizes_musd=_base_pp.sizes_musd,
        lambdas_per_day=_base_pp.lambdas_per_day * lam_scale,
        tiers=tiers,
        psi=_base_pp.psi,
        eta=eta * BP,
    )

    return ModelParams(
        currencies=["USD", "EUR"],
        ref_ccy="USD",
        sigma={"USD": 0.0, "EUR": sigma_eur * BP},
        corr={},
        k=_base.k,
        mu={"USD": 0.0, "EUR": 0.0},
        gamma=gamma,
        kappa=np.zeros((2, 2)),
        T_days=_base.T_days,
        pairs={("EUR", "USD"): pp},
    )


def _revenue_rate(mp, res, y):
    """Instantaneous expected revenue rate ($/day) at inventory y.

    R = sum over directions × tiers × sizes of: lambda_j * f(delta*) * delta* * z_j
    Converted from M$/day to $/day (* 1e6).
    """
    pp = mp.pairs[("EUR", "USD")]
    revenue = 0.0
    # Both directions: client buys EUR/sells USD, and client buys USD/sells EUR
    for ccy_pay, ccy_sell in [("EUR", "USD"), ("USD", "EUR")]:
        for t_idx, tier in enumerate(pp.tiers):
            for z, lam in zip(pp.sizes_musd, pp.lambdas_per_day):
                delta = res.markup(t_idx, ccy_pay, ccy_sell, z, y)
                f = logistic_f(delta, tier.alpha, tier.beta)
                revenue += lam * f * delta * z
    # delta is in decimal, z in M$ → revenue in M$/day (decimal), convert to $/day
    return revenue * 1e6


def evaluate_qois(params):
    """Solve ODE and return 6 QoIs.

    Set A (policy):  [tier spread diff (bps), inventory skew (bps), hedge rate (M$/day)]
    Set B (economic): [revenue rate at y=0 ($/day), hedging cost at y=10 ($/day),
                       net revenue at y=10 ($/day)]
    """
    mp = build_modified_params(params)
    res = run_multicurrency_mm(mp, n_steps=500)
    pp = mp.pairs[("EUR", "USD")]

    y_flat = np.zeros(2)
    y_long = np.array([0.0, 10.0])  # 10 M$ long EUR

    # --- Set A: policy QoIs ---
    # QoI 1: tier spread differential (tier 1 - tier 2) at y=0, z=1 M$, in bps
    delta_t1 = res.markup(0, "EUR", "USD", 1.0, y_flat)
    delta_t2 = res.markup(1, "EUR", "USD", 1.0, y_flat)
    tier_diff = (delta_t1 - delta_t2) / BP

    # QoI 2: inventory skew (tier 1, z=1) in bps
    delta_t1_long = res.markup(0, "EUR", "USD", 1.0, y_long)
    skew = (delta_t1_long - delta_t1) / BP

    # QoI 3: hedge rate at y=10 (M$/day)
    xi = res.hedge_rate("EUR", "USD", y_long)

    # --- Set B: economic QoIs ---
    # QoI 4: expected revenue rate at y=0 ($/day)
    R_flat = _revenue_rate(mp, res, y_flat)

    # QoI 5: hedging cost rate at y=10 ($/day)
    # L(xi) = psi|xi| + eta*xi^2, in decimal M$/day → convert to $/day
    hedge_cost = (pp.psi * abs(xi) + pp.eta * xi**2) * 1e6

    # QoI 6: net revenue rate at y=10 ($/day)
    R_long = _revenue_rate(mp, res, y_long)
    net_revenue = R_long - hedge_cost

    return np.array([tier_diff, skew, xi, R_flat, hedge_cost, net_revenue])

In [ ]:
# Sanity check: evaluate at nominal and verify n_steps=500 vs 2000
qoi_500 = evaluate_qois(NOMINAL)

# Compare with n_steps=2000 (higher accuracy)
mp_nom = build_modified_params(NOMINAL)
res_2000 = run_multicurrency_mm(mp_nom, n_steps=2000)
pp_nom = mp_nom.pairs[("EUR", "USD")]
y_flat = np.zeros(2)
y_long = np.array([0.0, 10.0])

d_t1 = res_2000.markup(0, "EUR", "USD", 1.0, y_flat)
d_t2 = res_2000.markup(1, "EUR", "USD", 1.0, y_flat)
d_t1_long = res_2000.markup(0, "EUR", "USD", 1.0, y_long)
xi_2000 = res_2000.hedge_rate("EUR", "USD", y_long)
R_flat_2000 = _revenue_rate(mp_nom, res_2000, y_flat)
R_long_2000 = _revenue_rate(mp_nom, res_2000, y_long)
hedge_cost_2000 = (pp_nom.psi * abs(xi_2000) + pp_nom.eta * xi_2000**2) * 1e6

qoi_2000 = np.array([
    (d_t1 - d_t2) / BP,
    (d_t1_long - d_t1) / BP,
    xi_2000,
    R_flat_2000,
    hedge_cost_2000,
    R_long_2000 - hedge_cost_2000,
])

print("QoIs at nominal parameter values:")
print(f"{'QoI':>45s}  {'n=500':>12s}  {'n=2000':>12s}  {'rel err':>10s}")
for name, v500, v2000 in zip(QOI_NAMES, qoi_500, qoi_2000):
    err = abs(v500 - v2000) / (abs(v2000) + 1e-30)
    print(f"{name:>45s}  {v500:12.4f}  {v2000:12.4f}  {err:10.2e}")

# Time a single evaluation
t0 = time.time()
for _ in range(100):
    evaluate_qois(NOMINAL)
t_per = (time.time() - t0) / 100
print(f"\nTime per evaluation: {t_per*1000:.1f} ms")
print(f"Estimated time for N=10000 Sobol: {t_per * 10000 * (N_PARAMS + 2) / 60:.1f} min")

## Sobol indices

Global sensitivity analysis using Saltelli's Monte Carlo estimator.

- First-order $S_i$: fraction of output variance due to parameter $i$ alone (Saltelli 2010)
- Total-effect $S_{Ti}$: includes all interactions involving parameter $i$ (Jansen 1999)

Total model evaluations: $N \times (k + 2)$ where $k = 8$ parameters.

In [33]:
def saltelli_sample(N, ranges, seed=42):
    """Generate two independent uniform sample matrices A, B scaled to parameter ranges."""
    rng = np.random.default_rng(seed)
    k = ranges.shape[0]
    lo, hi = ranges[:, 0], ranges[:, 1]
    A = lo + rng.random((N, k)) * (hi - lo)
    B = lo + rng.random((N, k)) * (hi - lo)
    return A, B


def evaluate_saltelli_samples(model_func, A, B):
    """Evaluate model at all Saltelli sample points.

    Returns f_A (N, n_qoi), f_B (N, n_qoi), f_C (k, N, n_qoi).
    """
    N, k = A.shape
    total = N * (k + 2)

    # Stack all parameter sets: A, B, C_0, ..., C_{k-1}
    all_params = np.empty((total, k))
    all_params[:N] = A
    all_params[N:2*N] = B
    for i in range(k):
        C_i = A.copy()
        C_i[:, i] = B[:, i]
        all_params[(2 + i) * N:(3 + i) * N] = C_i

    # Evaluate all samples
    n_qoi = len(model_func(A[0]))
    results = np.empty((total, n_qoi))

    for j in tqdm(range(total), desc="Saltelli evaluations", mininterval=2.0):
        results[j] = model_func(all_params[j])

    # Unpack
    f_A = results[:N]
    f_B = results[N:2*N]
    f_C = np.empty((k, N, n_qoi))
    for i in range(k):
        f_C[i] = results[(2 + i) * N:(3 + i) * N]

    return f_A, f_B, f_C


def compute_sobol_indices(f_A, f_B, f_C):
    """Compute first-order and total-effect Sobol indices.

    First-order (Saltelli 2010):  S_i  = mean(f_B * (f_Ci - f_A)) / Var(Y)
    Total-effect (Jansen 1999):  S_Ti = mean((f_A - f_Ci)^2) / (2 * Var(Y))
    """
    N, n_qoi = f_A.shape
    k = f_C.shape[0]

    f_all = np.concatenate([f_A, f_B], axis=0)
    var_Y = np.var(f_all, axis=0)

    S = np.empty((k, n_qoi))
    S_T = np.empty((k, n_qoi))

    for i in range(k):
        S[i] = np.mean(f_B * (f_C[i] - f_A), axis=0) / var_Y
        S_T[i] = np.mean((f_A - f_C[i])**2, axis=0) / (2.0 * var_Y)

    return S, S_T

In [34]:
N_SOBOL = 10_000

print(f"Sobol analysis: N={N_SOBOL}, k={N_PARAMS}, total evals={N_SOBOL*(N_PARAMS+2)}")
A, B = saltelli_sample(N_SOBOL, RANGES, seed=42)
f_A, f_B, f_C = evaluate_saltelli_samples(evaluate_qois, A, B)

Sobol analysis: N=10000, k=8, total evals=100000


Saltelli evaluations: 100%|██████████| 100000/100000 [12:22<00:00, 134.75it/s]


In [ ]:
S, S_T = compute_sobol_indices(f_A, f_B, f_C)

for label, indices in [("First-order S_i", S), ("Total-effect S_Ti", S_T)]:
    print(f"{label}:")
    header = "  ".join(f"{'QoI '+str(q+1):>8s}" for q in range(N_QOIS))
    print(f"{'':>15s}  {header}")
    for i in range(N_PARAMS):
        vals = "  ".join(f"{indices[i, q]:8.3f}" for q in range(N_QOIS))
        print(f"{PARAM_LABELS[i]:>15s}  {vals}")
    print()

print(f"Sum of S_i:  {S.sum(axis=0)}  (should be close to 1 if interactions are small)")
print(f"Sum of S_Ti: {S_T.sum(axis=0)}  (>1 indicates interactions)")

In [ ]:
x = np.arange(N_PARAMS)
width = 0.35

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
set_labels = ["Set A: Policy", "Set B: Economic"]

for q in range(N_QOIS):
    row = q // 3
    col = q % 3
    ax = axes[row, col]
    ax.bar(x - width/2, S[:, q], width, label=r'$S_i$ (first-order)', color='steelblue')
    ax.bar(x + width/2, S_T[:, q], width, label=r'$S_{Ti}$ (total-effect)', color='coral')
    ax.set_ylabel('Sobol index')
    ax.set_title(QOI_NAMES[q], fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(PARAM_NAMES, rotation=45, ha='right', fontsize=9)
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1.05)

fig.suptitle(f'Sobol Indices (N = {N_SOBOL})', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Forward Uncertainty Quantification

Sample parameters from uniform priors, propagate through the ODE model,
and estimate the distribution of each quantity of interest via kernel density estimation.

In [ ]:
# Reuse the Sobol A and B evaluations as forward UQ samples (2N = 20,000 points)
fwd_qois = np.concatenate([f_A, f_B], axis=0)
N_FWD = fwd_qois.shape[0]
print(f"Forward UQ using {N_FWD} samples (reused from Sobol computation)")

nominal_qois = evaluate_qois(NOMINAL)

print(f"\n{'QoI':>50s}  {'mean':>10s}  {'std':>10s}  {'5%':>10s}  {'95%':>10s}  {'nominal':>10s}")
for q in range(N_QOIS):
    data = fwd_qois[:, q]
    print(f"{QOI_NAMES[q]:>50s}  {data.mean():10.2f}  {data.std():10.2f}  "
          f"{np.percentile(data, 5):10.2f}  {np.percentile(data, 95):10.2f}  "
          f"{nominal_qois[q]:10.2f}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for q in range(N_QOIS):
    row = q // 3
    col = q % 3
    ax = axes[row, col]
    data = fwd_qois[:, q]

    # KDE with Silverman bandwidth
    kde = gaussian_kde(data, bw_method='silverman')
    x_grid = np.linspace(data.min(), data.max(), 300)
    ax.plot(x_grid, kde(x_grid), 'k-', linewidth=1.5, label='PDF')
    ax.axvline(nominal_qois[q], color='r', linestyle='--', linewidth=1.5, label='Nominal')

    # Shade 90% CI
    p5, p95 = np.percentile(data, [5, 95])
    mask = (x_grid >= p5) & (x_grid <= p95)
    ax.fill_between(x_grid[mask], kde(x_grid[mask]), alpha=0.2, color='steelblue', label='90% CI')

    ax.set_xlabel(QOI_NAMES[q])
    ax.set_ylabel('Density')
    ax.set_title(QOI_NAMES[q], fontsize=11)
    ax.legend(fontsize=8)

fig.suptitle(f'Forward UQ (N = {N_FWD})', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()